In [ ]:
%reload_ext autoreload
%autoreload 2
from importlib import reload

import os
import sys

import logging
import warnings
import numpy as np
import astropy as ap
import scipy as sp
import scipy.stats
import matplotlib as mpl
import matplotlib.pyplot as plt

import h5py
import tqdm.notebook as tqdm

import kalepy as kale
import kalepy.utils
import kalepy.plot

import holodeck as holo
import holodeck.sams
import holodeck.gravwaves
from holodeck import cosmo, utils, plot, discrete, sams, host_relations, _PATH_DATA
from holodeck.constants import MSOL, PC, YR, MPC, GYR, SPLC
from pathlib import Path

import compare_discrete

# Silence annoying numpy errors
np.seterr(divide='ignore', invalid='ignore', over='ignore')
warnings.filterwarnings("ignore", category=UserWarning)

# Plotting settings
mpl.rc('font', **{'family': 'serif', 'sans-serif': ['Times'], 'size': 15})
mpl.rc('lines', solid_capstyle='round')
mpl.rc('mathtext', fontset='cm')
plt.rcParams.update({'grid.alpha': 0.5})
mpl.style.use('default')   # avoid dark backgrounds from dark theme vscode

log = holo.log
log.setLevel(logging.INFO)

# ---- Define filepath containing simulation galaxy merger data files ----#
# ---- (if using files not in _PATH_DATA) ---- #
_HOME_PATH = Path('~/').expanduser()
p = os.path.join(_HOME_PATH, 'cosmo_sim_merger_data')
if os.path.exists(p):
    _SIM_MERGER_PATH = p
else:
    p = os.path.join(_HOME_PATH, 'nanograv/cosmo_sim_merger_data')
    if os.path.exists(p):
        _SIM_MERGER_PATH = p
    else:
        _SIM_MERGER_PATH = _PATH_DATA
print(f"{_SIM_MERGER_PATH=}")
# ------------------------------------------------------------------------ #

### Functions in this notebook
- calc_sim_gpf_and_gmr() --- calculate galaxy pair frac and merger rate from sims, output those along with m, q, z bins
- compare_gal_merger_dens() --- Calculate space density of galaxy mergers for sims and sams and compare. Optional cuts on m and q.
- compare_gal_merger_rate() --- Compare galaxy merger rate between sims and sams. Optional cuts on m and q.
- rg15_merger_rates() --- Returns Rodriguez-Gomez+2015 merger rates from analytic fitting functions.
- plot_rg15_merger_rates() --- Plot Rodriguez-Gomez+2015 merger rates using calculation in holodeck sam module and my manual calculation to check. 

In [ ]:
def calc_sim_gpf_and_gmr(dpop, basePath, mask=None, verbose=False,
                         mres_factor=0.0):    # mass_type='tot', nbins=10,  
    """Calculate galaxy pair frac and merger rate from sims, output those along with m, q, z bins. Optional cuts on m, q, and z."""
    
    print(f"in calc_sim_gpf_and_gmr: {dpop.lbl=}, {basePath=}")
    
    data = dpop.pop.mbulge/MSOL
    dp_box_vol_mpc = dpop.pop._sample_volume / (1.0e6*PC)**3

    # try:
    #    fpmass = dpop.pop.first_prog_mass # first progenitor mass at tmax (time of max past mass)
    #    npmass = dpop.pop.next_prog_mass # next progenitor mass at tmax (time of max past mass)
    prog_q_at_tmax = dpop.pop.prog_mass_ratio  # progenitor mass ratio at tmax (time of max past mass)
    #mrg_dp_mstar_tot = dpop.pop.mstar_tot[:,2] # using the total stellar mass of descendant like RG15
    mrg_dp_mstar_desc = dpop.pop.mstar_desc # using the total stellar mass of descendant like RG15
        
    mrg_dp_qstar = prog_q_at_tmax
    mrg_dp_mstar = mrg_dp_mstar_desc
    print("\n\n\nDefining progenitor stellar mass ratio at tmax and descendant total mass at descendant snap as in RG15.")

    # ---- at least for now, we will stick with the RG15 mass definitions and add these alternates back in if needed.
    #except:
    #    mrg_dp_mstar_all = dpop.pop.mbulge # this should be the total stellar mass of the descendant subhalo not the bulge mass of progs
    #    mrg_dp_mstar1 = np.max(mrg_dp_mstar_all, axis=1) 
    #    mrg_dp_mstar2 = np.min(mrg_dp_mstar_all, axis=1)
    #    mrg_dp_mstar_tot = mrg_dp_mstar1 + mrg_dp_mstar2
    #    tmp = mrg_dp_mstar1 + mrg_dp_mstar2
    #    #print(f"{tmp.min()=}, {tmp.max()=}, {tmp.shape=}")
    #    mrg_dp_qstar = mrg_dp_mstar2 / mrg_dp_mstar1
        
    #    #if mask is not None:
    #    #    data = data[mask]
    #    #    if verbose: print(f"after mask, in calc_gal_mrg_rate: {data.shape=}")

    #    ## NOTE that none of these are actually identical to the definition in RG15, 
    #    ## which calculates the merger rate as a function of *descendant* stellar mass
    #    ## and *progenitor* stellar mass ratio. `mass_type` = 'tot' will be closest, but 
    #    ## we might want to code up an option to use this exact defn in the future.
    #    if mass_type == 'tot':
    #        mrg_dp_mstar = mrg_dp_mstar_tot
    #    elif mass_type == 'pri':
    #        mrg_dp_mstar = mrg_dp_mstar1
    #    elif mass_type == 'all':
    #        mrg_dp_mstar = mrg_dp_mstar_all
    #    else:
    #        err = "`mass_type` must be 'tot', 'pri', or 'all'"
    #        raise ValueError(err)
    #    print("Defining progenitor stellar mass ratio at prog snap and total mass as sum of prog masses (DIFFERENT than RG15).")

    if verbose: 
        print(f"{mrg_dp_qstar.min()=}, {mrg_dp_qstar.max()=}, {mrg_dp_qstar.mean()=}, "
              f"{np.median(mrg_dp_qstar)=}, {mrg_dp_qstar.shape=}")
        print(f"{mrg_dp_mstar.min()=}, {mrg_dp_mstar.max()=}, {mrg_dp_mstar.mean()=}, "
              f"{np.median(mrg_dp_mstar)=}, {mrg_dp_mstar.shape=}\n\n\n")
    
    mrg_dp_redz = 1.0/dpop.pop.scafa - 1
    
    if mres_factor > 0:
        print(f"\n\n{mrg_dp_mstar.shape=}, {mrg_dp_qstar.shape=}, {mrg_dp_redz.shape=}")
        print(f"REMOVING mergers with Mdesc or Mdesc*q/(1+q) below {mres_factor*dpop.mres_baryon:.4g} msun")
        mrg_dp_mstar_msun = mrg_dp_mstar/MSOL
        # use mdesc and q to calculate an effective mass of the secondary galaxy, compare to mass limit
        m2_eff = mrg_dp_mstar_msun * mrg_dp_qstar / (1 + mrg_dp_qstar) 
        mask = ((mrg_dp_mstar_msun >= mres_factor * dpop.mres_baryon)&
                (m2_eff >= mres_factor * dpop.mres_baryon))
        mrg_dp_redz = mrg_dp_redz[mask]
        mrg_dp_qstar = mrg_dp_qstar[mask]
        mrg_dp_mstar_msun = mrg_dp_mstar_msun[mask]
        mrg_dp_mstar = mrg_dp_mstar[mask]
        print(f"After removing unresolved mergers:")
        print(f"{mrg_dp_mstar.shape=}, {mrg_dp_qstar.shape=}, {mrg_dp_redz.shape=}\n\n")

              
    
    for s in ['Illustris-1', 'TNG50-1', 'TNG100-1', 'TNG300-1']:
        if s in dpop.lbl: 
            print(f"{s=}")
            gsmf_fpath = os.path.join(basePath, f"gsmf_all_snaps_{s}.hdf5")
            print(f"{gsmf_fpath=}")
            break
    try:                
        f = h5py.File(gsmf_fpath, "r")
        try:
            grpname=(f"gas-{dpop.minNparts[0]:03d}_dm-{dpop.minNparts[1]:03d}_"
                     f"star-{dpop.minNparts[4]:03d}_bh-{dpop.minNparts[5]:03d}")
            grp=f[grpname]
        except:
            raise Exception(f"Could not open group in GSMF file: {grpname}")

    except:
        raise Exception(f"Could not open GSMF file {gsmf_fpath}.")

    snapnums = f.attrs['SnapshotNums']
    dlgm_all_snaps = f.attrs['LogMassBinWidth']
    snap_scalefacs = f.attrs['SnapshotScaleFacs']
    mbinedgs_all_snaps = np.array(grp['StellarMassBinEdges'])
    mhist_all_snaps = np.array(grp['StellarMassHistograms'])
    zsnaps = 1.0 / snap_scalefacs - 1.0

    ###mhist_all_snaps = np.zeros((mbinedgs_all_snaps.size-1, snapnums.size)) 

    ## NOTE: streamline this if we're always going to recalculate the mass histogram here anyway
    #for i,n in enumerate(snapnums):
    #    print(f"loading subhalo lens & masses for snap {n}...")
    #    Nparts = f[f'Snap{n:03d}']['SubhaloLenType']
    #    masses = f[f'Snap{n:03d}']['SubhaloMassType']
    #    if Nparts.shape[0] == 0:
    #        continue
        
    #    print(f"before mask: {Nparts.shape=}, {masses.shape=}")
    #    mask = ((Nparts[:,0]>dpop.minNparts[0])&(Nparts[:,1]>dpop.minNparts[1])&
    #            (Nparts[:,4]>dpop.minNparts[4])&(Nparts[:,5]>dpop.minNparts[5]))
    #    Nparts = Nparts[mask,:]
    #    masses = masses[mask,:]
    #    print(f"before mask: {Nparts.shape=}, {masses.shape=}")
    #    mstar = masses[:,4]
    #    mhist_all_snaps[:, i], tmp = np.histogram(np.log10(mstar), bins=mbinedgs_all_snaps)


    #ncombine_mbins = 40
    #ncombine_mbins = 10
    ncombine_mbins = 5
    #ncombine_mbins = 2
    #ncombine_mbins = 20
    dlgm_new = dlgm_all_snaps * ncombine_mbins
    mbin_edges = mbinedgs_all_snaps[::ncombine_mbins]
    mbin_edges = np.append(mbin_edges, mbin_edges[-1]+dlgm_new)
    mbin_edges = mbin_edges[mbin_edges >= 8.0]
    if verbose:
        print(f"{mhist_all_snaps.shape=}, {mhist_all_snaps.min()=}, {mhist_all_snaps.max()=}")
        print(f"{snapnums=}")
        print(f"{dlgm_all_snaps=}, {mbinedgs_all_snaps.shape=}, {mhist_all_snaps.shape=}")
        #print(f"{mbins_all_snaps=}")
        print(f"{mbin_edges=}")
    
    #dq = 0.05
    #qbin_edges = np.arange(0,1+dq,dq)
    dlgq = 0.2
    #dlgq = 0.1
    #dlgq = 0.05
    #dlgq = 0.4
    #dlgq = 0.8
    #dlgq = 0.5
    #dlgq = 1.0
    #lgqbin_edges = np.arange(-4, 0+dlgq, dlgq)
    lgqbin_edges = np.arange(-3-dlgq, 0+dlgq, dlgq)
    qbin_edges = 10**lgqbin_edges
    
    #dz = 0.5
    dz = 0.25
    #dz = 0.125
    #dz = 1.0
    #dz = 2.0
    #zbin_edges = np.arange(0,20+dz,dz)
    zbin_edges = np.arange(0,4+dz,dz)
    
    if verbose:
        print("BEFORE cuts:")
        print(f"lgm min/max: {np.log10(mrg_dp_mstar.min()/MSOL)=}, {np.log10(mrg_dp_mstar.max()/MSOL)=}")
        print(f"q min/max: {mrg_dp_qstar.min()=}, {mrg_dp_qstar.max()=}")
        print(f"z min/max: {mrg_dp_redz.min()=}, {mrg_dp_redz.max()=}")

    ix_mmin = np.where(mbin_edges<np.log10(mrg_dp_mstar.min()/MSOL))[0]
    ix_mmax = np.where(mbin_edges>np.log10(mrg_dp_mstar.max()/MSOL))[0]
    ix_qmin = np.where(qbin_edges<mrg_dp_qstar.min())[0]
    ix_qmax = np.where(qbin_edges>mrg_dp_qstar.max())[0]
    ix_zmin = np.where(zbin_edges<mrg_dp_redz.min())[0]
    ix_zmax = np.where(zbin_edges>mrg_dp_redz.max())[0]

    if verbose:
        print(f"{mbin_edges=}, {np.log10(mrg_dp_mstar.min()/MSOL)=}, "
              f"{np.log10(mrg_dp_mstar.max()/MSOL)=}")
        print(ix_mmin, ix_mmax, mbin_edges.size)
        print(f"{qbin_edges=}, {mrg_dp_qstar.min()=}, {mrg_dp_qstar.max()=}")
        print(ix_qmin, ix_qmax, qbin_edges.size)
        print(f"{zbin_edges=}, {mrg_dp_redz.min()=}, {mrg_dp_redz.max()=}")
        print(ix_zmin, ix_zmax, zbin_edges.size)

    if len(ix_mmax) > 0 and ix_mmax[0] != mbin_edges.size-1:
        mbin_edges = mbin_edges[:ix_mmax[0]+1]
    if len(ix_mmin) > 0 and ix_mmin[-1] != 0:
        mbin_edges = mbin_edges[ix_mmin[-1]:]
    if len(ix_zmax) > 0 and ix_zmax[0] != zbin_edges.size-1:
        zbin_edges = zbin_edges[:ix_zmax[0]+1]
    if len(ix_zmin) > 0 and ix_zmin[-1] != 0:
        zbin_edges = zbin_edges[ix_zmin[-1]:]
    
    if verbose:
        print("AFTER cuts:")
        print(f"lgm min/max: {np.log10(mrg_dp_mstar.min()/MSOL)=}, {np.log10(mrg_dp_mstar.max()/MSOL)=}")
        print(f"q min/max: {mrg_dp_qstar.min()=}, {mrg_dp_qstar.max()=}")
        print(f"z min/max: {mrg_dp_redz.min()=}, {mrg_dp_redz.max()=}")
        print(f"{mbin_edges=}, {np.log10(mrg_dp_mstar.min()/MSOL)=}, {np.log10(mrg_dp_mstar.max()/MSOL)=}")
        print(ix_mmin, ix_mmax, mbin_edges.size)
        print(f"{qbin_edges=}, {mrg_dp_qstar.min()=}, {mrg_dp_qstar.max()=}")
        print(ix_qmin, ix_qmax, qbin_edges.size)
        print(f"{zbin_edges=}, {mrg_dp_redz.min()=}, {mrg_dp_redz.max()=}")
        print(ix_zmin, ix_zmax, zbin_edges.size)

    mqz_hist, tmp = np.histogramdd(np.array([np.log10(mrg_dp_mstar/MSOL), mrg_dp_qstar, mrg_dp_redz]).T, 
                                       bins=(mbin_edges, qbin_edges, zbin_edges))
    #mqz_hist, (mbin_edges, qbin_edges, zbin_edges) = np.histogramdd(np.array([np.log10(mrg_dp_mstar/MSOL), 
    #                                                                          mrg_dp_qstar, 
    #                                                                          mrg_dp_redz]).T, bins=nbins)
    
    
    sparse_count = 0
    zero_count = 0
    for i in range(mbin_edges.size-1):
        for j in range(qbin_edges.size-1):                
            for k in range(zbin_edges.size-1):
                if mqz_hist[i,j,k] == 0:
                    zero_count +=1

                if mqz_hist[i,j,k] < 5 and mqz_hist[i,j,k]>0:
                    sparse_count += 1
                    
                    #mqz_hist[i,j,k] = 0
                    print(f"mqz_hist[{i},{j},{k}]={mqz_hist[i,j,k]} for {mbin_edges[i]:.3g}<m<{mbin_edges[i+1]:.3g},"
                            f"{qbin_edges[j]:.3g}<q<{qbin_edges[j+1]:.3g}, {zbin_edges[k]}<z<{zbin_edges[k+1]}")

    
    if verbose:
        print(f"in gmr calculation: {sparse_count=}, {zero_count=}, "
              f"tot bins: {(mbin_edges.size-1)*(qbin_edges.size-1)*(zbin_edges.size-1)}")
        print(f"{mqz_hist.shape=}, {mbin_edges.shape=}, {qbin_edges.shape=}, {zbin_edges.shape=}")
        print(f"{mbin_edges=}")
        print(f"{qbin_edges=}")
        print(f"{zbin_edges=}")
        print(f"{mbinedgs_all_snaps=}")
        print(f"{mhist_all_snaps.shape=}, {mhist_all_snaps.min()=}, {mhist_all_snaps.max()=}")

    ## now load galaxy stellar masses and redshifts for each snapshot
    ## and calculate the avg number of galaxies per snapshot in each bin
    
    Ngal_per_snap = np.zeros((mbin_edges.size-1, snapnums.size))
    avg_Ngal_per_zbin = np.zeros((mbin_edges.size-1, zbin_edges.size-1))
    if verbose:
        print(f"{zsnaps=}")
        print(f"{zbin_edges=}")
        print(f"looping thru snaps for {basePath=}")
        
    for i,n in enumerate(snapnums):            
        for j in range(mbin_edges.size-1):
            Ngal_per_snap[j,i] = mhist_all_snaps[(mbinedgs_all_snaps[:-1]>=mbin_edges[j])&
                                                 (mbinedgs_all_snaps[1:]<=mbin_edges[j+1]), i].sum()

    for k in range(zbin_edges.size-1):
        thisbin = Ngal_per_snap[:, (zsnaps>=zbin_edges[k])&(zsnaps<zbin_edges[k+1])]
        if thisbin.shape[1] > 0:
            avg_Ngal_per_zbin[:,k] = thisbin.sum(axis=1)/thisbin.shape[1]     # CHECK THIS! 
        else:
            if verbose: print(f"skipping empty zbin {zbin_edges[k]}-{zbin_edges[k+1]}")
            
    if verbose:
        print(f"{Ngal_per_snap.shape=}")
        print(f"{avg_Ngal_per_zbin.shape=}")
        print(f"{Ngal_per_snap=}")
        print(f"{avg_Ngal_per_zbin=}")

    
    gpf = np.zeros_like(mqz_hist)
    for i in range(qbin_edges.size-1):
        # dNmrg(M*,q*,z) / dqstar: 
        for k in range(zbin_edges.size-1):
            for l in range(mbin_edges.size-1):
                if mqz_hist[l,i,k] > 0 and avg_Ngal_per_zbin[l,k] == 0:
                    print(f"***WARNING*** nonzero merger hist with zero galaxy hist "
                          f"(m{mbin_edges[l]:.3g},q{qbin_edges[i]:.3g},z{zbin_edges[k]})")
                if avg_Ngal_per_zbin[l,k] > 0:
                    gpf[l,i,k] = mqz_hist[l,i,k] / avg_Ngal_per_zbin[l,k]
                if gpf[l,i,k] > 1.0:
                    print(f'***WARNING*** invalid {gpf[l,i,k]=} at '
                          f'm{mbin_edges[l]:.3g},q{qbin_edges[i]:.3g},z{zbin_edges[k]};\n'
                          f'{mqz_hist[l,i,k]=}, {avg_Ngal_per_zbin[l,k]=}')
        #gpf[:,i,:] = mqz_hist[:,i,:] / avg_Ngal_per_zbin
        ## this gives us the pair fraction -- can also compare this! is it defined the same way in the sam???
    if verbose: print(f"{gpf.shape=}")
    
    ## then divide by time interval spanned by z bin, convert from yr to Gyr
    tlook_zbin_edges = cosmo.z_to_tlbk(zbin_edges) / GYR
    dt_zbins = tlook_zbin_edges[1:] - tlook_zbin_edges[:-1]
    if verbose:
        print(f"{zbin_edges=}")
        print(f"{tlook_zbin_edges=}")
        print(f"{dt_zbins=}")
        print(f"{dt_zbins.shape=}, {dt_zbins.min()=}, {dt_zbins.max()=}")
        print(f"{dt_zbins.all()}")
    if dt_zbins.any() < 0.0: 
        raise ValueError("`dt_zbins` must be positive-valued!")

    gal_mrg_rate = gpf / dt_zbins

    ## divide by the width of the mass ratio bins to get merger rate per unit mass ratio
    if verbose: print(f"{gal_mrg_rate.shape=}")

    dqbins = qbin_edges[1:] - qbin_edges[:-1]
    if verbose: print(f"{dqbins=}")
    for i in range(dqbins.size):
        gal_mrg_rate[:,i,:] = gal_mrg_rate[:,i,:] / dqbins[i] 

    if verbose: 
        print(f"{gal_mrg_rate.shape=}")
        print(f"\n\nin gpf gmr calculation:")
        print(f"{gpf.shape=}, {gal_mrg_rate.shape=}, {dt_zbins.shape=}")
        print(f"{dt_zbins=}")
        #print(f"{gpf[0,0,:]=}, {gal_mrg_rate[0,0,:]=}\n")
        #print(f"{gpf[0,1,:]=}, {gal_mrg_rate[0,1,:]=}\n")
        #print(f"{gpf[1,0,:]=}, {gal_mrg_rate[1,0,:]=}\n")
        #print(f"{gpf[1,1,:]=}, {gal_mrg_rate[1,1,:]=}\n\n")

    return gpf, gal_mrg_rate, mqz_hist, avg_Ngal_per_zbin, dt_zbins, (mbin_edges, qbin_edges, zbin_edges)


In [ ]:
def compare_gal_merger_dens(dpops, colors=None, lws=None, labels=None, 
                            sam_compare=None, gpf_flag=None, mcut=None, qcut=None,
                            legend=True):
    """Calculate space density of galaxy mergers for sims and sams and compare. Optional cuts on m and q."""
    
    fig = plt.figure(figsize=(14,5))
    ax1 = fig.add_subplot(131) 
    ax1.set_xlabel(r'$\log_{10}(m_{*1})$')
    ax1.set_ylabel(r'$\frac{d\eta_{gal-gal}}{d\log_{10}m_{*1}}$ [Mpc$^{-3}$]')

    ax2 = fig.add_subplot(132) 
    ax2.set_xlabel(r'$q_*$')
    ax2.set_ylabel(r'$d\eta_{gal-gal}$ / $dq_{*1}$ [Mpc$^{-3}$]')
    ax2.set_xscale('log')

    ax3 = fig.add_subplot(133) 
    ax3.set_xlabel('redshift')
    ax3.set_ylabel(r'$d\eta_{gal-gal}$ / $dz$ [Mpc$^{-3}$]')

    ls_arr = ['-.','-']
    lw_arr = [2,3]  
    color_arr = ['m','k']

    if sam_compare is not None:
        assert gpf_flag is not None, "gpf_flag must be defined if sam_compare is not None."
        
        if (not isinstance(sam_compare,list)) and (not isinstance(sam_compare,tuple)):
            sam_compare = [sam_compare]
        if (not isinstance(gpf_flag,list)) and (not isinstance(gpf_flag,tuple)):
            gpf_flag = [gpf_flag]
            
        for i,s in enumerate(sam_compare):

            lbl = 'gpf' if gpf_flag[i] else 'gmr'
            print(f"{i=}, {gpf_flag[i]=}, {lbl=}")
            
            mstar_pri, mstar_rat, mstar_tot, redz = s.mass_stellar() 
            #mstar = mstar_pri
            mstar = mstar_tot
            mstar_arr = mstar[:,-1,0]
            mstar_rat_arr = mstar_rat[0,:,0]
            redz_arr = redz[0,0,:]
            print(f"{mstar.shape=}, {mstar_rat.shape=}, {redz.shape=}")
            print(f"{mstar_arr.shape=}, {mstar_rat_arr.shape=}, {redz_arr.shape=}")
                
            if mcut is not None:
                ixm = np.where(mstar_arr>=mcut)[0][0]
                print(f"mass cut index ={ixm}")
                mstar_arr = mstar_arr[ixm:]
                mstar = mstar[ixm:,:,:]
                mstar_rat = mstar_rat[ixm:,:,:]
                redz = redz[ixm:,:,:]
                print(f"imposed mass cut >{mcut/MSOL:.4g} msun. "
                      f"new mass array shape: {mstar_arr.shape}")

            if qcut is not None:
                ixq = np.where(mstar_rat_arr >= qcut)[0][0]
                mstar_rat_arr = mstar_rat_arr[ixq:]
                mstar = mstar[:,ixq:,:]
                mstar_rat = mstar_rat[:,ixq:,:]
                redz = redz[:,ixq:,:]
                print(f"imposed mass ratio cut >{qcut} "
                      f"new mass ratio array shape: {mstar_rat_arr.shape}")
            
            print(f"{mstar.shape=}, {mstar_rat.shape=}, {redz.shape=}")
            print(f"{mstar_arr.shape=}, {mstar_rat_arr.shape=}, {redz_arr.shape=}")
                
            print(f"{mstar.shape=}, {mstar_rat.shape=}, {redz.shape=}")

            nd = s._ndens_gal(mstar, mstar_rat, redz) ## dnt need cuts bc passed arrays to it that already had
            
            print(f"{nd.shape=},{nd.min()=},{nd.max()=}")
            
            integ = utils.trapz(nd, np.log10(mstar_arr/MSOL), axis=0, cumsum=False) # int over primary mass with mrat=1
            print(f"integ over mstar_arr: {integ.shape}")
            sam_dngalgal_dz = utils.trapz(integ, mstar_rat_arr, axis=1, cumsum=False) # int over mstar_rat (same for all mstar_pri)
            print(f"integ over mstar_rat_arr: {sam_dngalgal_dz.shape}")
            sam_dngalgal_dz = sam_dngalgal_dz.sum(axis=0)
            print(f"sum over mstar_arr: {sam_dngalgal_dz.shape}")
            sam_dngalgal_dz = sam_dngalgal_dz.sum(axis=0)
            print(f"sum over mstar_rat_arr: {sam_dngalgal_dz.shape}")
            print(f"total sam_dngalgal (from dz): {sam_dngalgal_dz.sum()}")

            integ = utils.trapz(nd, mstar_rat_arr, axis=1, cumsum=False)
            sam_dngalgal_dmstar = utils.trapz(integ, redz_arr, axis=2, cumsum=False)
            sam_dngalgal_dmstar = sam_dngalgal_dmstar.sum(axis=1)
            sam_dngalgal_dmstar = sam_dngalgal_dmstar.sum(axis=1)
            print(f"total sam_dngalgal (from dmstar): {sam_dngalgal_dmstar.sum()}")

            integ = utils.trapz(nd, np.log10(mstar_arr/MSOL), axis=0, cumsum=False)
            sam_dngalgal_dmstar_rat = utils.trapz(integ, redz_arr, axis=2, cumsum=False)
            sam_dngalgal_dmstar_rat = sam_dngalgal_dmstar_rat.sum(axis=0)
            sam_dngalgal_dmstar_rat = sam_dngalgal_dmstar_rat.sum(axis=1)
            print(f"total sam_dngalgal (from dmstar_rat): {sam_dngalgal_dmstar_rat.sum()}")

            ax1.plot(np.log10(mstar_arr/MSOL), sam_dngalgal_dmstar,lw=lw_arr[i],
                     ls=ls_arr[i], color=color_arr[i],label=lbl)
            ax2.plot(mstar_rat_arr, sam_dngalgal_dmstar_rat,lw=lw_arr[i],
                     ls=ls_arr[i], color=color_arr[i],label=lbl)
            ax3.plot(redz_arr, sam_dngalgal_dz,lw=lw_arr[i],
                     ls=ls_arr[i], color=color_arr[i],label=lbl)

    for i,dp in enumerate(dpops):
        dp_box_vol_mpc = dp.pop._sample_volume / (1.0e6*PC)**3

        dp_mstar = dp.pop.mbulge
        dp_mstar1 = np.max(dp_mstar, axis=1)
        dp_mstar2 = np.min(dp_mstar, axis=1)
        dp_mstar_tot = dp_mstar1 + dp_mstar2
        dp_qstar = dp_mstar2 / dp_mstar1
        
        dp_redz = 1.0/dp.pop.scafa - 1
        dp_dngalgal_tot = redz.size / dp_box_vol_mpc # total galaxy merger number density in cMpc^-3
        print(f'{dp_dngalgal_tot=}')
        dp_z_hist, dp_z_bin_edges = np.histogram(dp_redz, bins=20)
        dp_z_binsize = dp_z_bin_edges[1:] - dp_z_bin_edges[:-1]
        dp_dngalgal_dz = dp_z_hist / dp_z_binsize / dp_box_vol_mpc
        
        #lgm1_hist, lgm1_bin_edges = np.histogram(np.log10(mstar1/MSOL), bins=20)
        dp_lgm1_hist, dp_lgm1_bin_edges = np.histogram(np.log10(dp_mstar_tot/MSOL), bins=20)
        dp_lgm1_binsize = dp_lgm1_bin_edges[1:] - dp_lgm1_bin_edges[:-1]
        dp_dngalgal_dlog10m1 = dp_lgm1_hist / dp_lgm1_binsize / dp_box_vol_mpc

        dp_qstar_hist, dp_qstar_bin_edges = np.histogram(dp_qstar, bins=20)
        dp_qstar_binsize = dp_qstar_bin_edges[1:] - dp_qstar_bin_edges[:-1]
        dp_dngalgal_dqstar = dp_qstar_hist / dp_qstar_binsize / dp_box_vol_mpc
        print(f'dngalgal_tot = {dp_dngalgal_tot/dp_qstar_binsize[0]/dp_lgm1_binsize[0]/dp_z_binsize[0]}')

        ax1.set_ylim(1.0e-6,0.5)
        ax1.set_yscale('log')
        ax2.set_yscale('log')
        ax3.set_yscale('log')
        ax1.plot(dp_lgm1_bin_edges[:-1], dp_dngalgal_dlog10m1, label=dp.lbl)
        ax2.plot(dp_qstar_bin_edges[:-1], dp_dngalgal_dqstar, label=dp.lbl)
        ax3.plot(dp_z_bin_edges[:-1], dp_dngalgal_dz, label=dp.lbl)

    if legend:
        ax1.legend()
        ax2.legend()
        ax3.legend()
    fig.subplots_adjust(wspace=0.5)


In [ ]:
def gmr_rebin(bin_edges, gpf_1d, gmr_1d, hist_1d, Ngal_1d, 
              dt_zbins_1d, bin_type='m', verbose=False):
    
    if bin_type == 'm' or bin_type == 'q':

        if verbose:
            print(f"input variables:")
            print(f"{bin_edges.shape=}, {gpf_1d.shape=}, {gmr_1d.shape=}, {hist_1d.shape=}, "
                  f"{Ngal_1d.shape=}, {dt_zbins_1d.shape=}")
            print(f"{bin_edges=}")
            print(f"{gpf_1d=}")
            print(f"{gmr_1d=}")
            print(f"{hist_1d=}")
            print(f"{Ngal_1d=}")
            print(f"{dt_zbins_1d=}")

        if hist_1d.max() == 0:
            msg = 'WARNING: no rebinning for empty histogram {hist_1d=}. returning original values.'
            log.warning(msg)
            warnings.warn(msg)
            return bin_edges, hist_1d, Ngal_1d, gpf_1d
        
        ix_first_nonzero = np.min(np.where(hist_1d>0)[0])
        
        if verbose: print(f"{ix_first_nonzero=}")
        new_bin_edges = np.array([])
        new_hist = np.array([])
        new_Ngal = np.array([])
        tmp_hist_arr = np.array([])
        tmp_Ngal_arr = np.array([])
        
        for i in range(hist_1d.size):
            # rebin hist1d
            
            if i<ix_first_nonzero:
                if verbose: print(f'{i=}: skipping bin {bin_edges[i]}-{bin_edges[i+1]} with hist_1d[{i}]={hist_1d[i]}.')
                continue
            elif i==ix_first_nonzero:
                if verbose: print(f'{i=}: found first nonzero bin (left edge {bin_edges[i]}) with hist_1d[{i}]={hist_1d[i]}.')
                new_bin_edges = np.append(new_bin_edges, bin_edges[i]) # i gives left edge of bin
                
            if tmp_hist_arr.sum() + hist_1d[i] >= 10:
                if verbose: print(f"{i=}: defining new right edge of bin {bin_edges[i+1]}"
                                  f" with {(tmp_hist_arr.sum()+hist_1d[i])=}.")
                new_bin_edges = np.append(new_bin_edges, bin_edges[i+1]) # i+1 gives right edge of bin 
                new_hist = np.append(new_hist, tmp_hist_arr.sum() + hist_1d[i])
                new_Ngal = np.append(new_Ngal, (tmp_Ngal_arr.sum() + Ngal_1d[i])/(tmp_Ngal_arr.size+1))
                tmp_hist_arr = np.array([])
                tmp_Ngal_arr = np.array([])
            else:
                tmp_hist_arr = np.append(tmp_hist_arr, hist_1d[i])
                tmp_Ngal_arr = np.append(tmp_Ngal_arr, hist_1d[i])
                if tmp_hist_arr.sum() >= 10:
                    if verbose: print(f"{i=}: defining new right edge of bin {new_bin_edges[-1]}-{bin_edges[i+1]}"
                                      f"with {(tmp_hist_arr.sum()+hist_1d[i])=}.")
                    new_bin_edges = np.append(new_bin_edges, bin_edges[i+1]) # i+1 gives right edge of bin 
                    new_hist = np.append(new_hist, tmp_hist_arr.sum() + hist_1d[i])
                    new_Ngal = np.append(new_Ngal, (tmp_Ngal_arr.sum() + Ngal_1d[i])/(tmp_Ngal_arr.size+1))
                    tmp_hist_arr = np.array([])
                    tmp_Ngal_arr = np.array([])

        if new_bin_edges[-1] != bin_edges[-1] and hist_1d[-1] > 0:
            msg = (f'WARNING: discarding elements of last bin with {hist_1d[-1]} elements:'
                   f'{bin_edges=} \n{hist_1d=}'
                   f'{new_bin_edges=} \n{new_hist=}')
            log.warning(msg)
            warnings.warn(msg)
            
        new_gpf = new_hist / new_Ngal

        if verbose:
            print(f"new variables")
            print(f"{new_bin_edges.shape=}, {new_hist.shape=}, {new_Ngal.shape=}, {new_gpf.shape=}")
            print(f"{new_bin_edges=}")
            print(f"{new_hist=}")
            print(f"{new_Ngal=}")
            print(f"{new_gpf=}")
        
        return new_bin_edges, new_hist, new_Ngal, new_gpf
    
    elif bin_type == 'z':
        #do something more complicated
        raise ValueError('bin_type `z` not coded yet')
    else:
        raise ValueError(f"Invalid `bin_type` keyword value. Must be `m`, `q`, or `z`.")


def compare_gal_merger_rate(dpops, colors=None, lws=None, labels=None, sam_compare=None,
                            gpf_flag=None, mcut=None, qcut=None, mres_factor=10.0, legend=True, 
                            filename='compare_gmr.png', verbose=False):
    """Compare galaxy merger rate between sims and sams. Optional cuts on m and q."""
    
    #fig = plt.figure(figsize=(12,3))
    fig, axs = plt.subplots(3,3,figsize=(12,12))  #plt.subplots(nrows, ncols, **kwargs)

    # gmr vs descendant mass
    mlbl = r'$\log_{10}(m_{*1})$'
    mxlims = (7.5,12.5)
    mylims = (1.0e-3,1.0e5)

    # gmr vs mass ratio
    qlbl = r'$q_*$'
    qxlims = (1.0e-4,1.0)
    qylims = (1.0e-3,1.0e5)

    # gmr vs z
    zlbl = '1+z'
    zxlims = (1,9)
    zylims = (1.0e-3,1.0e5)

    for i in range(3):
        axs[0][i].set(yscale='log',xlabel=mlbl, xlim=mxlims, ylim=mylims)
        axs[1][i].set(xscale='log',yscale='log',xlabel=qlbl, xlim=qxlims, ylim=qylims)
        axs[2][i].set(xscale='log',yscale='log',xlabel=zlbl, xlim=zxlims, ylim=zylims)


    if sam_compare is not None:
        assert gpf_flag is not None, "gpf_flag must be defined if sam_compare is not None."
        
        if (not isinstance(sam_compare,list)) and (not isinstance(sam_compare,tuple)):
            sam_compare = [sam_compare]
        if (not isinstance(gpf_flag,list)) and (not isinstance(gpf_flag,tuple)):
            gpf_flag = [gpf_flag]

        lw_arr = [2,3]  
        ls_arr = ['-','-']
        color_arr = ['k','k']

        tgt_m = np.array([1.0e9,1.0e10,1.0e11])
        tgt_q = np.array([1.0e-3,1.0e-2,0.1,0.25])
        tgt_z = np.array([0.1,1.0,2.0])

        qcolor = ['orange','darkorange','red','darkred']
        mcolor = ['cyan','green','blue']
        zcolor = ['thistle','violet','magenta']
        alpha_arr = [1.0,0.75,0.5,0.25]
        
        # ---- PLOT SAM DATA ----
        for i,s in enumerate(sam_compare):
    
            lbl = 'gpf' if gpf_flag[i] else 'gmr'
            if verbose: print(f"{i=}, {gpf_flag[i]=}, {lbl=}")

            mstar_pri, mstar_rat, mstar_tot, redz = s.mass_stellar() 
            mstar = mstar_tot
            
            target_m_ix = np.zeros_like(tgt_m).astype('int64')
            for n in range(len(tgt_m)):
                target_m_ix[n] = get_nearest_index(mstar[:,0,0]/MSOL, tgt_m[n])

            target_q_ix = np.zeros_like(tgt_q).astype('int64')
            for n in range(len(tgt_q)):
                target_q_ix[n] = get_nearest_index(mstar_rat[0,:,0], tgt_q[n])
    
            target_z_ix = np.zeros_like(tgt_z).astype('int64')
            for n in range(len(tgt_z)):
                target_z_ix[n] = get_nearest_index(redz[0,0,:], tgt_z[n])
            
            gal_merger_rate = s._gal_merger_rate(mstar, mstar_rat, redz) * GYR # convert from s^-1 to Gyr^-1   
                
            if verbose: 
                print(f"\n\n{mstar.shape=},{mstar_rat.shape=},{redz.shape=}")
                print(f'{s._gal_merger_rate(mstar, mstar_rat, redz).shape=}')
                print(gal_merger_rate.min(),gal_merger_rate.max())
        

            for j in range(target_q_ix.size):
                lbl3 = 'SAM' if (i == 0 and j == target_q_ix.size-2) else None


                # ---- Plot merger rate vs m for selected target z values (one per plot) 
                # ----     and target q values (all on each plot)
                for n in range(target_z_ix.size):
                    axs[0][n].plot(np.log10(mstar_tot[:,target_q_ix[j],target_z_ix[n]]/MSOL), 
                              gal_merger_rate[:,target_q_ix[j],target_z_ix[n]],
                              ls=ls_arr[i], lw=0.5*j+1.5,color='m') #,label=lbl) 

                # ---- Plot merger rate vs (1+z) for selected target m values (one per plot) 
                # ----     and target q values (all on each plot)
                for k in range(target_m_ix.size):
                    axs[2][k].plot(redz[target_m_ix[k], target_q_ix[j], :], 
                                   gal_merger_rate[target_m_ix[k], target_q_ix[j], :],
                                   ls=ls_arr[i],lw=0.5*j+0.5,color=color_arr[i]) #,label=lbl)    

                
            for k in range(target_m_ix.size):
                # ---- Plot merger rate vs q for selected target z values (one per plot) 
                # ----     and target m values (all on each plot)
                for n in range(target_z_ix.size):
                    axs[1][n].plot(mstar_rat[target_m_ix[k], :, target_z_ix[n]],
                                   gal_merger_rate[target_m_ix[k], :, target_z_ix[n]],
                                   ls=ls_arr[i],lw=0.5*k+0.5,color=color_arr[i]) #,label=lbl)    
            
    # ---- PLOT SIMULATION DATA ----
    ls_arr = ['-','--','-.',':']*2
    for i,dp in enumerate(dpops):
        
        tmp = calc_sim_gpf_and_gmr(dp, _PATH_DATA, mres_factor=mres_factor, verbose=verbose)
        sim_gpf, sim_gmr, sim_mqz_hist, sim_Ngal, sim_dt_zbins, (sim_mbin_edges, sim_qbin_edges, sim_zbin_edges) = tmp

        sim_mbin_edges = 10.0**sim_mbin_edges * MSOL

        
        sim_mbin_centers = sim_mbin_edges[:-1] + 0.5 * (sim_mbin_edges[1:]-sim_mbin_edges[:-1])
        sim_qbin_centers = sim_qbin_edges[:-1] + 0.5 * (sim_qbin_edges[1:]-sim_qbin_edges[:-1])
        sim_zbin_centers = sim_zbin_edges[:-1] + 0.5 * (sim_zbin_edges[1:]-sim_zbin_edges[:-1])

        sim_lgmbin_edges_msun = np.log10(sim_mbin_edges/MSOL)
        sim_lgmbin_centers_msun = (sim_lgmbin_edges_msun[:-1] + 
                                   0.5 * (sim_lgmbin_edges_msun[1:] - sim_lgmbin_edges_msun[:-1]))

        if verbose:
            print(f"\n\nLOOPING OVER DP. {dp.basepath=}\n\n")
            print(f"{dp.lbl=}, {sim_gpf.shape=}, {sim_gmr.shape=}, "
                  f"{sim_mbin_edges.shape=}, {sim_zbin_edges.shape=}, {sim_zbin_edges.shape=}")
        
            print(f"{sim_mbin_edges=}")
            print(f"{sim_mbin_centers=}")
            print(f"{sim_lgmbin_edges_msun=}")
            print(f"{sim_lgmbin_centers_msun=}")

        dp_box_vol_mpc = dp.pop._sample_volume / (1.0e6*PC)**3

        
        sim_target_m_ix = np.zeros_like(tgt_m).astype('int64')
        for n in range(len(tgt_m)):
            sim_target_m_ix[n] = get_bin_index(sim_mbin_edges/MSOL, tgt_m[n])

        sim_target_q_ix = np.zeros_like(tgt_q).astype('int64')
        for n in range(len(tgt_q)):
            sim_target_q_ix[n] = get_bin_index(sim_qbin_edges, tgt_q[n])
    
        sim_target_z_ix = np.zeros_like(tgt_z).astype('int64')
        for n in range(len(tgt_z)):
            sim_target_z_ix[n] = get_bin_index(sim_zbin_edges, tgt_z[n])

        if verbose:
            print(f"{tgt_m=}, {sim_target_m_ix=} ")
            print(f"{tgt_q=}, {sim_target_q_ix=} ")                
            print(f"{tgt_z=}, {sim_target_z_ix=} ")

        for j in range(sim_target_q_ix.size):
            lbl1 = f"q{tgt_q[j]}" if i == 0 else None
            lbl2 = dp.lbl  if j == sim_target_q_ix.size-2 else None
            lbl3 = 'Sim' if (i == 0 and j == sim_target_q_ix.size-2) else None


            # ---- Plot merger rate vs m for selected target z values (one per plot) 
            # ----     and target q values (all on each plot)
            for n in range(sim_target_z_ix.size):
                
                ##################### TEST ###################
                this_dq = (sim_qbin_edges[sim_target_q_ix[j]+1]-sim_qbin_edges[sim_target_q_ix[j]])
                if verbose:
                    print("\n\n\n***   REBIN TEST   ***")
                    print(f"BEFORE REBIN: {sim_mbin_edges=}\n")
                    print(f"{sim_lgmbin_edges_msun=}\n"
                          f"sim_gpf={sim_gpf[:,sim_target_q_ix[j],sim_target_z_ix[n]]}\n"
                          f"sim_gmr={sim_gmr[:,sim_target_q_ix[j],sim_target_z_ix[n]]}\n"
                          f"sim_dt_zbins={sim_dt_zbins[sim_target_z_ix[n]]} "
                          f"{this_dq=} (target q ={tgt_q[j]}, "
                          f"qbin={sim_qbin_edges[sim_target_q_ix[j]+1]},{sim_qbin_edges[sim_target_q_ix[j]]}\n"
                          f"for q={sim_qbin_centers[sim_target_q_ix[j]]} ({sim_target_q_ix[j]=}), z={sim_zbin_centers[sim_target_z_ix[n]]} ({sim_target_z_ix[n]=})")
                tmp = gmr_rebin(sim_mbin_edges, sim_gpf[:,sim_target_q_ix[j],sim_target_z_ix[n]],
                                sim_gmr[:,sim_target_q_ix[j],sim_target_z_ix[n]], 
                                sim_mqz_hist[:,sim_target_q_ix[j],sim_target_z_ix[n]],
                                sim_Ngal[:,sim_target_z_ix[n]], sim_dt_zbins[sim_target_z_ix[n]],
                                bin_type='m',verbose=False)
                new_mbin_edges, new_hist, new_Ngal, new_gpf = tmp
                new_gmr = new_gpf / sim_dt_zbins[sim_target_z_ix[n]] 
                new_gmr /= (sim_qbin_edges[sim_target_q_ix[j]+1]-sim_qbin_edges[sim_target_q_ix[j]])
                new_lgmbin_edges_msun = np.log10(new_mbin_edges/MSOL)
                new_lgmbin_centers_msun = (new_lgmbin_edges_msun[:-1] + 
                                           0.5*( new_lgmbin_edges_msun[1:] - new_lgmbin_edges_msun[:-1]))
                if verbose:
                    print(f"AFTER REBIN: {new_mbin_edges=}\n")
                    print(f"{new_lgmbin_edges_msun=}\n"
                          f"{new_gpf=}\n"
                          f"{new_gmr=}")
                axs[0][n].plot(new_lgmbin_centers_msun, new_gmr,
                               color=qcolor[j],lw=0.5*j+1, ls=ls_arr[i], alpha=alpha_arr[i], label=lbl1)
                axs[0][n].plot(sim_lgmbin_centers_msun, sim_gmr[:,sim_target_q_ix[j],sim_target_z_ix[n]],
                               color=qcolor[j],lw=0.5*j+0.5, ls=':', alpha=alpha_arr[i], label=lbl1)

            # ---- Plot merger rate vs (1+z) for selected target m values (one per plot) 
            # ----     and target q values (all on each plot)
            for k in range(sim_target_m_ix.size):
                axs[2][k].plot(sim_zbin_centers, sim_gmr[sim_target_m_ix[k], sim_target_q_ix[j], :],
                               color=qcolor[j],lw=0.5*j+0.5, ls=ls_arr[i],  alpha=alpha_arr[i],label=lbl1)            
            
        for k in range(sim_target_m_ix.size):
            lbl4 = f"m{np.log10(tgt_m[k])}" if i == 0 else None
            # ---- Plot merger rate vs q for selected target z values (one per plot) 
            # ----     and target m values (all on each plot)
            for n in range(sim_target_z_ix.size):
                axs[1][n].plot(sim_qbin_centers, sim_gmr[sim_target_m_ix[k], :, sim_target_z_ix[n]],
                               color=mcolor[k],lw=0.5*k+0.5, ls=ls_arr[i],  alpha=alpha_arr[i],label=lbl4)    
        

    fsz = 9
    for i in range(3):
        axs[0][i].title.set_text(f'z~{tgt_z[i]}')
        axs[1][i].title.set_text(f'z~{tgt_z[i]}')
        axs[2][i].title.set_text(f'm~{np.log10(tgt_m[i])}')
        for j in range(3):
            axs[j][i].legend(fontsize=fsz)
        
    fig.subplots_adjust(hspace=0.3)
    
    fig.savefig(filename,dpi=300)

In [ ]:
def rg15_merger_rate(Mtot, mu, z):
    """Returns Rodriguez-Gomez+2015 merger rates from analytic fitting functions."""
    
    M0 = 2.0e11 * MSOL
    A0 = 10.0**-2.2287 #± 0.0045
    eta = 2.4644 #± 0.0128
    alpha0 = 0.2241 #± 0.0038
    alpha1 = -1.1759 #± 0.0316
    beta0 = -1.2595 #± 0.0026
    beta1 = 0.0611 #± 0.0021
    gamma = -0.0477 #± 0.0013
    delta0 = 0.7668 #± 0.0202
    delta1 = -0.4695 #± 0.0440

    Az = A0 * (1 + z)**eta
    alphaz = alpha0 * (1 + z)**alpha1
    betaz = beta0 * (1 + z)**beta1
    deltaz = delta0 * (1 + z)**delta1
    
    gmr = Az * (Mtot / (1.0e10*MSOL))**alphaz * (1 + (Mtot/M0)**deltaz) 
    gmr = gmr * mu**(betaz + gamma*np.log10(Mtot / (1.0e10*MSOL)))
    
    return gmr

In [ ]:
def get_nearest_index(arr,val):
    diff = np.abs(arr-val)
    index =  np.where(diff==diff.min())[0][0]
    return index

def get_bin_index(bin_edges, val):

    if val<bin_edges.min() or val>bin_edges.max():
        raise ValueError(f'{val=} out of range ({bin_edges.min()},{bin_edges.max()})')

    index = np.where((val>=bin_edges[:-1])&(val<bin_edges[1:]))[0][0]

    if index.size != 1:
        raise ValueError(f'{index=}: no valid bin for {val=} in {bin_edges=}')

    print(f"in get_bin_index: {val=}, {index=}, {bin_edges=}")
    
    return index
    
    
def plot_rg15_merger_rates(sam):
    """Plot Rodriguez-Gomez+2015 merger rates using calculation in holodeck sam module and my manual calculation to check."""
    mstar_pri, mstar_rat, mstar_tot, redz = sam.mass_stellar() 
    rg15_gmr = rg15_merger_rate(mstar_tot, mstar_rat, redz)


    mu0pt25_ix = get_index(mstar_rat[0,:,0],0.25)
    mu0pt01_ix = get_index(mstar_rat[0,:,0],0.01)
    m9_ix = get_index(mstar_tot[:,0,0],1.0e9*MSOL)
    m10_ix = get_index(mstar_tot[:,0,0],1.0e10*MSOL)
    m11_ix = get_index(mstar_tot[:,0,0],1.0e11*MSOL)
    z0pt1_ix = get_index(redz[0,0,:],0.1)
    print(z0pt1_ix, mu0pt25_ix, mu0pt01_ix)
    #print(mstar_rat[-1,:,0])
    #print(redz[0,0,:])
    #print(mstar_tot[:,0,0]/MSOL)
    #print(mstar_tot[:,-1,0]/MSOL)
    print(rg15_gmr.shape)
    print(mstar_tot.shape)
    print(sam.mrat)
    print(mstar_rat[0,:,0])

    sam_gmr = sam._gal_merger_rate(mstar_tot, mstar_rat, redz) * GYR # convert from s^-1 to Gyr^-1
    print(f"sam_gmr shape: {sam_gmr.shape}")

    fig = plt.figure(figsize=(12,5))
    ax1 = fig.add_subplot(131)
    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.set_xlabel('stellar mass')
    ax1.set_ylabel('Galaxy merger rate [Gyr^-1]')
    ax1.plot(mstar_tot[:,0,z0pt1_ix]/MSOL, rg15_gmr[:,0,z0pt1_ix], label=f'mu={mstar_rat[0,0,0]:.4g}')
    ax1.plot(mstar_tot[:,mu0pt01_ix,z0pt1_ix]/MSOL, rg15_gmr[:,mu0pt01_ix,z0pt1_ix],
             label=f'mu={mstar_rat[0,mu0pt01_ix,0]:.4g}')
    ax1.plot(mstar_tot[:,mu0pt25_ix,z0pt1_ix]/MSOL, rg15_gmr[:,mu0pt25_ix,z0pt1_ix],
             label=f'mu={mstar_rat[0,mu0pt25_ix,0]:.4g}')

    ax1.plot(mstar_tot[:,0,z0pt1_ix]/MSOL, sam_gmr[:,0,z0pt1_ix],'--')
    ax1.plot(mstar_tot[:,mu0pt01_ix,z0pt1_ix]/MSOL, sam_gmr[:,mu0pt01_ix,z0pt1_ix],'--')
    ax1.plot(mstar_tot[:,mu0pt25_ix,z0pt1_ix]/MSOL, sam_gmr[:,mu0pt25_ix,z0pt1_ix],'--')
    
    ax1.legend()

    ax2 = fig.add_subplot(132)
    ax2.set_xscale('log')
    ax2.set_yscale('log')
    ax2.set_xlabel('stellar mass ratio')
    ax2.set_ylabel('Galaxy merger rate [Gyr^-1]')
    ax2.plot(mstar_rat[0,:,z0pt1_ix],rg15_gmr[0,:,z0pt1_ix],label=f'M={mstar_tot[0,0,0]/MSOL:.4g}')
    ax2.plot(mstar_rat[m9_ix,:,z0pt1_ix],rg15_gmr[m9_ix,:,z0pt1_ix],
             label=f'M={mstar_tot[m9_ix,0,0]/MSOL:.4g}')
    ax2.plot(mstar_rat[m10_ix,:,z0pt1_ix], rg15_gmr[m10_ix,:,z0pt1_ix],
             label=f'M={mstar_tot[m10_ix,0,0]/MSOL:.4g}')
    ax2.plot(mstar_rat[m11_ix,:,z0pt1_ix], rg15_gmr[m11_ix,:,z0pt1_ix],
             label=f'M={mstar_tot[m11_ix,0,0]/MSOL:.4g}')

    ax2.plot(mstar_rat[0,:,z0pt1_ix], sam_gmr[0,:,z0pt1_ix],'--')
    ax2.plot(mstar_rat[m9_ix,:,z0pt1_ix], sam_gmr[m9_ix,:,z0pt1_ix],'--')
    ax2.plot(mstar_rat[m10_ix,:,z0pt1_ix], sam_gmr[m10_ix,:,z0pt1_ix],'--')
    ax2.plot(mstar_rat[m11_ix,:,z0pt1_ix], sam_gmr[m11_ix,:,z0pt1_ix],'--')

    ax2.legend()

    print(f"{redz[m9_ix,mu0pt01_ix,:].shape=}")
    print(f"{rg15_gmr[m9_ix,mu0pt01_ix:,:].shape=}")
    print(f"{mstar_rat[m9_ix,mu0pt01_ix:,:].shape=}")
    print(utils.trapz(rg15_gmr[m9_ix,mu0pt01_ix:,:],mstar_rat[0,mu0pt01_ix:,0],axis=0,cumsum=False).shape)
    print(utils.trapz(rg15_gmr[m9_ix,mu0pt01_ix:,:],mstar_rat[0,mu0pt01_ix:,0],axis=0,cumsum=False).sum(axis=0).shape)
    #rg15_gmr_cum_mu = 
    #sam_dngalgal_dz = utils.trapz(integ, mstar_rat_arr, axis=1, cumsum=False)                
    #sam_dngalgal_dz = sam_dngalgal_dz.sum(axis=0)
    ax3 = fig.add_subplot(133)
    ax3.set_xlim(1,10)
    ax3.set_xscale('log')
    ax3.set_yscale('log')
    ax3.set_xlabel('1+z')
    ax3.set_ylabel('Galaxy merger rate [Gyr^-1]')
    ax3.plot(1+redz[m11_ix,mu0pt01_ix,:],
             utils.trapz(rg15_gmr[m11_ix,mu0pt01_ix:,:],mstar_rat[0,mu0pt01_ix:,0],axis=0,cumsum=False).sum(axis=0),
             label=f'M={mstar_tot[m11_ix,mu0pt01_ix,0]/MSOL:.4g},mu>={mstar_rat[m11_ix,mu0pt01_ix,0]:.4g}')
    ax3.plot(1+redz[m11_ix,mu0pt25_ix,:],
             utils.trapz(rg15_gmr[m11_ix,mu0pt25_ix:,:],mstar_rat[0,mu0pt25_ix:,0],axis=0,cumsum=False).sum(axis=0),
             label=f'M={mstar_tot[m11_ix,mu0pt25_ix,0]/MSOL:.4g},mu>={mstar_rat[m11_ix,mu0pt25_ix,0]:.4g}')
    ax3.plot(1+redz[m10_ix,mu0pt25_ix,:],
             utils.trapz(rg15_gmr[m10_ix,mu0pt25_ix:,:],mstar_rat[0,mu0pt25_ix:,0],axis=0,cumsum=False).sum(axis=0),
             label=f'M={mstar_tot[m10_ix,mu0pt25_ix,0]/MSOL:.4g},mu>={mstar_rat[m10_ix,mu0pt25_ix,0]:.4g}')
    ax3.plot(1+redz[m9_ix,mu0pt25_ix,:],
             utils.trapz(rg15_gmr[m9_ix,mu0pt25_ix:,:],mstar_rat[0,mu0pt25_ix:,0],axis=0,cumsum=False).sum(axis=0),
             label=f'M={mstar_tot[m9_ix,mu0pt25_ix,0]/MSOL:.4g},mu>={mstar_rat[m9_ix,mu0pt25_ix,0]:.4g}')

    ax3.plot(1+redz[m11_ix,mu0pt01_ix,:],
             utils.trapz(sam_gmr[m11_ix,mu0pt01_ix:,:],
                         mstar_rat[0,mu0pt01_ix:,0],axis=0,cumsum=False).sum(axis=0),'--')
    ax3.plot(1+redz[m11_ix,mu0pt25_ix,:],
             utils.trapz(sam_gmr[m11_ix,mu0pt25_ix:,:],
                         mstar_rat[0,mu0pt25_ix:,0],axis=0,cumsum=False).sum(axis=0),'--')
    ax3.plot(1+redz[m10_ix,mu0pt25_ix,:],
             utils.trapz(sam_gmr[m10_ix,mu0pt25_ix:,:],
                         mstar_rat[0,mu0pt25_ix:,0],axis=0,cumsum=False).sum(axis=0),'--')
    ax3.plot(1+redz[m9_ix,mu0pt25_ix,:],
             utils.trapz(sam_gmr[m9_ix,mu0pt25_ix:,:],
                         mstar_rat[0,mu0pt25_ix:,0],axis=0,cumsum=False).sum(axis=0),'--')

    #ax3.plot(1+redz[m10_idx,mu0pt01_idx,:],rg15_gmr[m10_idx,mu0pt01_idx,:],
    #         label=f'M={mstar_tot[m10_idx,mu0pt01_idx,0]/MSOL:.4g},mu>={mstar_rat[m10_idx,mu0pt01_idx,0]:.4g}')
    #ax3.plot(1+redz[m11_idx,mu0pt01_idx,:],rg15_gmr[m11_idx,mu0pt01_idx,:],
    #         label=f'M={mstar_tot[m11_idx,mu0pt01_idx,0]/MSOL:.4g},mu>={mstar_rat[m11_idx,mu0pt01_idx,0]:.4g}')
    #ax3.plot(redz[m9_idx,:,z0pt1_idx],rg15_gmr[m9_idx,:,z0pt1_idx],
    #         label=f'M={mstar_tot[m9_idx,0,z0pt1_idx]/MSOL:.4g}')
    #ax2.plot(mstar_rat[m10_idx,:,z0pt1_idx], rg15_gmr[m10_idx,:,z0pt1_idx],
    #         label=f'M={mstar_tot[m10_idx,0,z0pt1_idx]/MSOL:.4g}')
    #ax2.plot(mstar_rat[m11_idx,:,z0pt1_idx], rg15_gmr[m11_idx,:,z0pt1_idx],
    #         label=f'M={mstar_tot[m11_idx,0,z0pt1_idx]/MSOL:.4g}')
    #ax1.plot(mstar_tot[:,-1,0]/MSOL,rg15_gmr[:,-1,0],label=f'mu={mstar_rat[0,-1,0]:.4g}')

    ax3.legend()
    fig.suptitle('Galaxy merger rates at z=0.1') 
    
    return

In [ ]:
# ---- Define the GWB frequencies
freqs, freqs_edges = utils.pta_freqs()
print(f"{freqs.shape[0]=}, {freqs_edges.shape[0]=}")
NFREQS = freqs.shape[0]
NREALS = 100
NLOUD = 10
#NLOUD = 1

# ---- Create discrete population(s) (including a rescaled version of TNG300, rTN300
#tmp = compare_discrete.create_dpops(allow_mbh0=True, mod_mmbulge=True, skip_evo=False, fsa_only=False, 
#                                    inclRescale=True, nreals=NREALS, nloudest=NLOUD, fpath=_SIM_MERGER_PATH)
#all_dpops, tng_dpops, all_fsa_dpops, tng_fsa_dpops = tmp
tmp = compare_discrete.create_dpops(allow_mbh0=True, mod_mmbulge=True, skip_evo=False, fsa_only=True, 
                                    inclRescale=True, nreals=NREALS, nloudest=NLOUD, bfrac=[0.5,1.0], 
                                    subhalo_mstar_defn='MaxPastMass', fpath=_SIM_MERGER_PATH)
                                    #subhalo_mstar_defn='SubhaloMassType', fpath=_SIM_MERGER_PATH)
all_dpops, tng_dpops, all_fsa_dpops, tng_fsa_dpops = tmp

In [ ]:
#print(f"(nfreq, nloud, nreals): {all_fsa_dpops[0].gwb.sspar[0].shape}") #nfreq, nloud, nreals

print(f"Inspiral timescale for all binaries = {all_fsa_dpops[0].tau / GYR} Gyr")

print("all_dpops:")
for d in all_dpops: print(d.lbl)
print("tng_dpops:")
for d in tng_dpops: print(d.lbl)
print("all_fsa_dpops:")
for d in all_fsa_dpops: print(d.lbl)
print("tng_fsa_dpops:")
for d in tng_fsa_dpops: print(d.lbl)

In [ ]:
print("creating SAM using Galaxy Pair Fraction (GPF) + Galaxy Merger Timescale (GMT)...")

sam = sams.Semi_Analytic_Model(gpf = sams.GPF_Power_Law())

print("    ...calculating hardening")
hard = holo.hardening.Fixed_Time_2PL_SAM(sam, all_fsa_dpops[0].tau, sepa_init=1.0e4*PC)

#print("    ...creating gwb")
#gwb_sam = sam.gwb(freqs_edges, hard, realize=NREALS, loudest=NLOUD, params=True)

#gwb_sam_L100 = sam.gwb(freqs_edges, hard, realize=50, loudest=100, params=True)

# this is what we've been using as the default for the sam GWB,
# until we had to switch to get the param output
#gwb_sam = sam.gwb_new(freqs_edges, hard, realize=50)
gwb_new_sam = sam.gwb_new(freqs_edges, hard, realize=NREALS)  ### ***NOTE*** this returns hc, not hc2

In [ ]:
print("creating 'no-GPF' SAM using Galaxy Merger Rate (GMR) "
      "    (uses galaxy merger rates directly from RG15 instead of GPF+GMT)...")

sam_no_gpf = sams.Semi_Analytic_Model()

print("    ...calculating hardening for no-GPF SAM")
hard_no_gpf = holo.hardening.Fixed_Time_2PL_SAM(sam_no_gpf, all_fsa_dpops[0].tau, sepa_init=1.0e4*PC)
print("    ...creating gwb for no-GPF SAM")
gwb_sam_no_gpf = sam_no_gpf.gwb_new(all_fsa_dpops[0].freqs_edges, hard_no_gpf, realize=50)

# Galaxy merger rate plots

In [ ]:
##compare_gal_merger_rate(tng_dpops, sam_compare=[sam,sam_no_gpf],gpf_flag=[1,0], mcut=None)
##compare_gal_merger_rate(all_dpops, sam_compare=[sam,sam_no_gpf],gpf_flag=[1,0], mcut=None)
#compare_gal_merger_rate(all_fsa_dpops, sam_compare=[sam,sam_no_gpf],gpf_flag=[1,0], mcut=None)
#compare_gal_merger_rate(all_fsa_dpops, sam_compare=[sam,sam_no_gpf],gpf_flag=[1,0], mcut=None) #, verbose=True)


#compare_gal_merger_rate(ill_fsa_bh0_dpops, sam_compare=[sam_no_gpf],gpf_flag=[0], mcut=None) #, verbose=True)
#compare_gal_merger_rate(tng_fsa_dpops, sam_compare=[sam_no_gpf],gpf_flag=[0], mcut=None) #, filename='compare_gmr_ill.png') 
compare_gal_merger_rate(all_fsa_dpops, sam_compare=[sam_no_gpf],gpf_flag=[0], 
                        mcut=None, mres_factor=10.0) #, verbose=True) 


In [ ]:
plot_rg15_merger_rates(sam_no_gpf)

In [ ]:
x=np.arange(10)
y=np.arange(5,10,0.5)
print(np.array([x,y]).shape)
print(np.array([x,y]).T.shape)
print(np.array([x,y]).T)

### Compare galaxy merger densities and galaxy merger rates b/t sims & SAMs:

In [ ]:
##compare_gal_merger_dens([d for d in [dpop_tng100_1,dpop_tng100_2]],sam_compare=sam)
compare_gal_merger_dens(all_fsa_dpops, sam_compare=[sam,sam_no_gpf], #ill_hires_fid_and_fsa_dpops,
#compare_gal_merger_dens(all_hires_fid_and_fsa_dpops, sam_compare=[sam],
                        gpf_flag=[1,0],legend=True) #,  
##                        mcut=1.0e10*MSOL, qcut=0.1)
#compare_gal_merger_dens(all_hires_fid_and_fsa_dpops) #,  

In [ ]:
#compare_gal_merger_dens(t50_hires_fid_and_fsa_dpops, sam_compare=[sam,sam_no_gpf],
compare_gal_merger_dens(all_dpops, sam_compare=[sam,sam_no_gpf],
                        gpf_flag=[1,0],legend=True) 

In [ ]:
#compare_gal_merger_rate([d for d in [dpop_tng100_1,dpop_tng100_2]],sam_compare=sam)
compare_gal_merger_rate([], sam_compare=[sam,sam_no_gpf], gpf_flag=[1,0], legend=False) #,mcut=1.0e10*MSOL,qcut=0.1)


##compare_gal_merger_rate(tng_dpops, sam_compare=[sam,sam_no_gpf],gpf_flag=[1,0], mcut=None)
##compare_gal_merger_rate(all_dpops, sam_compare=[sam,sam_no_gpf],gpf_flag=[1,0], mcut=None)
#compare_gal_merger_rate(all_fsa_dpops, sam_compare=[sam,sam_no_gpf],gpf_flag=[1,0], mcut=None)
#compare_gal_merger_rate(all_fsa_dpops, sam_compare=[sam,sam_no_gpf],gpf_flag=[1,0], mcut=None)

# Black hole merger rate plots

In [ ]:
sam_redz_final, sam_rate, sam_fisco, sam_hc, sam_ndens = sam.rate_chirps(integrate=True)
sam_valid = (sam_redz_final > 0.0)
sam_redz_final[~sam_valid] = -1.0
print(sam_ndens[sam_valid].shape,sam_rate.shape,sam_valid.shape, sam_redz_final.shape, sam_redz_final[sam_valid].shape)
sam_mt, sam_mr, sam_rz = sam.edges
#plt.scatter(1/(1+redz_final[valid]),rate[valid[:-1,:-1,:-1]])
plt.plot(sam_rz[:-1],sam_rate.sum(axis=1).sum(axis=0)*YR)
print(sam_rate.sum()*YR)

In [ ]:
#plt.yscale('log')
#for d in all_dpops:
for d in all_fsa_dpops:
    if not hasattr(d, 'evo'):
        d.fixed = holo.hardening.Fixed_Time_2PL.from_pop(d.pop, d.tau)
        # Create an evolution instance using population and hardening mechanism
        print(f"creating evolution instance and evolving it...")
        d.evo = discrete.evolution.Evolution(d.pop, d.fixed)
        # evolve binary population
        d.evo.evolve()

    print(f"{d.lbl}: {d.evo.scafa.shape=}, {d.evo.sepa.shape=}, {d.evo.coal.shape=}, {d.evo.scafa[d.evo.coal,:].shape=}")
    print(f"{d.evo.coal[0]}")
    #for i in range(100):
    #    print(f"{d.evo.scafa[0,i]:.3g}, {d.evo.sepa[0,i]:.3g}")
    #print(f"{d.evo.sepa[d.evo.coal,-1].max():.4g}")
    #print(f"{d.evo.sepa[~d.evo.coal,-1].min():.4g}")
    #print(f"{d.evo.scafa[d.evo.coal,-1].max():.4g}")
    #print(f"{d.evo.scafa[~d.evo.coal,-1].min():.4g}")
    sim_scafa_final = d.evo.scafa[d.evo.coal,-1]
    sim_rz_final = 1.0/sim_scafa_final - 1.0
    #print(f"{d.pop.mass.shape=}, {d.evo.mass.shape=}")
    utils.mtmr_from_m1m2(d.pop.mass)
    sim_mt, sim_mr = utils.mtmr_from_m1m2(d.pop.mass[d.evo.coal,:])
    print(f"{sim_mt.shape=}, {sim_mt.min()=}, {sim_mt.max()=}, {sim_mr.min()=}, {sim_mr.max()=}")
    sim_dNdz, sim_rz_edg = np.histogram(sim_rz_final, bins=50) 
    sim_dNdz_massive_major, tmp = np.histogram(sim_rz_final[(sim_mt>1.0e9*MSOL)&(sim_mr>0.001)],bins=sim_rz_edg)
    sim_rz_bins = sim_rz_edg[:-1] + 0.5*(sim_rz_edg[1:] - sim_rz_edg[:-1])
    sim_ndens = sim_dNdz / d.pop._sample_volume_mpc3
    sim_ndens_massive_major = sim_dNdz_massive_major / d.pop._sample_volume_mpc3
    ###rate = np.zeros_like(rz_final)
    # get rest-frame dz/dt
    #dzdt = 1.0 / cosmo.dtdz(rz_final)
    sim_dzdt = 1.0 / cosmo.dtdz(sim_rz_bins)
    # get dVc/dz in units of [Mpc^3] to match `ndens` in units of [Mpc^-3]
    sim_dVcdz = cosmo.dVcdz(sim_rz_bins, cgs=False)

    # factor of 1/1+z to convert from rest-frame to observer-frame time-interval
    sim_rate = sim_ndens * sim_dzdt * sim_dVcdz / (1.0 + sim_rz_bins)
    sim_rate_massive_major = sim_ndens_massive_major * sim_dzdt * sim_dVcdz / (1.0 + sim_rz_bins)
    print(sim_rate.shape)
    plt.plot(sim_rz_bins, sim_rate*YR,label=d.lbl)
    plt.plot(sim_rz_bins, sim_rate_massive_major*YR,label=f"{d.lbl} massive major")
#plt.plot(sam_rz[:-1],sam_rate.sum(axis=1).sum(axis=0)*YR,label='SAM')
    
plt.legend()